In [1]:
"""
Phase 1 — Data Preprocessing for Hierarchical ECG Classifier
Outputs:
  - train_raw.npz      : windowed raw signals + labels + record_ids (for Stage 1 TCN)
  - test_raw.npz       : windowed raw signals + labels + record_ids (for Stage 1 TCN)
  - train_spectro.npz  : 3-channel spectrograms + labels + record_ids (for Stage 2 EfficientNet)
  - test_spectro.npz   : 3-channel spectrograms + labels + record_ids (for Stage 2 EfficientNet)
  - scaler.pkl         : fitted StandardScaler (reserved for NeuroKit features later)
"""

import numpy as np
import pandas as pd
import pywt
import pickle
from scipy.signal import butter, filtfilt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from skimage.transform import resize

In [2]:
signals = np.load("ECG_Processed_Dataset/ecg_signals.npy", allow_pickle=True)


In [3]:
signals.shape

(8528,)

## Basics

In [6]:
# ───────────────────────────────────────────────
# 0. CONFIG
# ───────────────────────────────────────────────
DATA_DIR         = "ECG_Processed_Dataset"          # folder containing ecg_meta.csv and ecg_signals.npy
OUT_DIR          = "New_Processed_Data"          # folder to write output .npz files

WINDOW_SIZE      = 4500
TRAIN_STEP       = 2250         # 50% overlap for train
TEST_STEP        = 4500         # non-overlapping for test
IMG_SIZE         = 224          # spectrogram resize target
RANDOM_STATE     = 42

LABEL_MAP        = {"N": 0, "A": 1, "O": 2, "~": 3}

# Augmentation targets (train only, non-Normal classes only)
# N=4040 → no augmentation
# A=590  → 1180 (2×)
# O=1965 → 3930 (2×)
# ~=227  → 454  (2×)
AUG_TARGETS      = {1: 2, 2: 2, 3: 2}   # label → multiplier




In [7]:
# ───────────────────────────────────────────────
# 1. LOAD DATA
# ───────────────────────────────────────────────
def load_data(data_dir):
    meta    = pd.read_csv(f"{data_dir}/ecg_meta.csv")
    signals = np.load(f"{data_dir}/ecg_signals.npy", allow_pickle=True)
    return meta, signals


# ───────────────────────────────────────────────
# 2. RECORD-LEVEL TRAIN/TEST SPLIT
# ───────────────────────────────────────────────
def split_records(meta, random_state=RANDOM_STATE):
    record_ids     = meta["record_id"].values
    record_labels  = meta["label"].values

    train_ids, test_ids = train_test_split(
        record_ids,
        test_size=0.2,
        stratify=record_labels,
        random_state=random_state
    )
    train_mask = meta["record_id"].isin(train_ids).values
    test_mask  = meta["record_id"].isin(test_ids).values
    return train_mask, test_mask




## BandPass filter

In [8]:
# ───────────────────────────────────────────────
# 3. BANDPASS FILTER + Z-SCORE
# ───────────────────────────────────────────────
def bandpass_filter(signal, lowcut=0.5, highcut=45.0, fs=300.0, order=2):
    nyq  = 0.5 * fs
    low  = lowcut  / nyq
    high = highcut / nyq
    b, a = butter(order, [low, high], btype="band")
    return filtfilt(b, a, signal)

def zscore_normalize(signal):
    mu  = signal.mean()
    std = signal.std()
    if std < 1e-8:
        return signal - mu
    return (signal - mu) / std

def preprocess_signal(signal):
    filtered = bandpass_filter(signal)
    return zscore_normalize(filtered)

## Windowing

In [9]:
# ───────────────────────────────────────────────
# 4. WINDOWING
# ───────────────────────────────────────────────
def extract_windows(signal, record_id, label, window_size, step):
    windows, labels, rids = [], [], []
    n = len(signal)
    start = 0
    while start + window_size <= n:
        windows.append(signal[start : start + window_size])
        labels.append(label)
        rids.append(record_id)
        start += step
    return windows, labels, rids

def build_windows(meta, signals, mask, window_size, step):
    all_windows, all_labels, all_rids = [], [], []
    indices = np.where(mask)[0]
    for idx in indices:
        row       = meta.iloc[idx]
        record_id = row["record_id"]
        label     = LABEL_MAP[row["label"]]
        signal    = preprocess_signal(signals[idx].astype(np.float32))
        w, l, r   = extract_windows(signal, record_id, label, window_size, step)
        all_windows.extend(w)
        all_labels.extend(l)
        all_rids.extend(r)
    return (
        np.array(all_windows, dtype=np.float32),
        np.array(all_labels,  dtype=np.int64),
        np.array(all_rids)
    )

## Data Augmentation

In [10]:
# ───────────────────────────────────────────────
# 5. AUGMENTATION  (CutMix + mild Gaussian noise)
#    Applied to train windows only, non-Normal classes only
# ───────────────────────────────────────────────
def cutmix_1d(sig_a, sig_b, alpha=0.3):
    """Randomly replace a contiguous segment of sig_a with the same segment from sig_b."""
    lam    = np.random.beta(alpha, alpha)
    length = int(WINDOW_SIZE * lam)
    start  = np.random.randint(0, WINDOW_SIZE - length + 1)
    out    = sig_a.copy()
    out[start : start + length] = sig_b[start : start + length]
    return out

def augment_windows(windows, labels, rids, aug_targets, random_state=RANDOM_STATE):
    rng = np.random.default_rng(random_state)
    aug_windows, aug_labels, aug_rids = [], [], []

    for label, multiplier in aug_targets.items():
        class_idx = np.where(labels == label)[0]
        n_orig    = len(class_idx)
        n_needed  = n_orig * multiplier - n_orig   # how many NEW samples to add

        for _ in range(n_needed):
            i     = rng.choice(class_idx)
            j     = rng.choice(class_idx)
            mixed = cutmix_1d(windows[i], windows[j])
            # mild Gaussian noise (std = 0.01 × signal std)
            noise = rng.normal(0, 0.01 * mixed.std(), size=mixed.shape).astype(np.float32)
            mixed = mixed + noise
            aug_windows.append(mixed)
            aug_labels.append(label)
            aug_rids.append(rids[i])

    if len(aug_windows) == 0:
        return windows, labels, rids

    aug_windows = np.array(aug_windows, dtype=np.float32)
    aug_labels  = np.array(aug_labels,  dtype=np.int64)
    aug_rids    = np.array(aug_rids)

    windows_out = np.concatenate([windows, aug_windows], axis=0)
    labels_out  = np.concatenate([labels,  aug_labels],  axis=0)
    rids_out    = np.concatenate([rids,    aug_rids],    axis=0)

    # shuffle
    perm = rng.permutation(len(windows_out))
    return windows_out[perm], labels_out[perm], rids_out[perm]

## Spectogram generation

In [ ]:
# ───────────────────────────────────────────────
# (3-channel)
#    Channel 1 : STFT magnitude (log-scale)
#    Channel 2 : CWT  (Morlet wavelet)
#    Channel 3 : Recurrence Plot
# ───────────────────────────────────────────────
def compute_stft_channel(signal, img_size=IMG_SIZE):
    from numpy.fft import fft
    # Short-time Fourier Transform via manual framing
    frame_size  = 256
    hop_size    = 64
    n_frames    = (len(signal) - frame_size) // hop_size + 1
    window      = np.hanning(frame_size)
    stft        = np.zeros((frame_size // 2, n_frames))
    for i in range(n_frames):
        start        = i * hop_size
        frame        = signal[start : start + frame_size] * window
        spectrum     = np.abs(fft(frame)[: frame_size // 2])
        stft[:, i]   = spectrum
    log_stft = np.log1p(stft).astype(np.float32)
    return resize(log_stft, (img_size, img_size), anti_aliasing=True).astype(np.float32)

def compute_cwt_channel(signal, img_size=IMG_SIZE):
    scales      = np.arange(1, 129)           # 128 scales
    coef, _     = pywt.cwt(signal, scales, "morl")
    magnitude   = np.abs(coef).astype(np.float32)
    return resize(magnitude, (img_size, img_size), anti_aliasing=True).astype(np.float32)

def compute_recurrence_channel(signal, img_size=IMG_SIZE, emb_dim=3, tau=1):
    # Phase-space embedding
    n        = len(signal)
    max_i    = n - (emb_dim - 1) * tau
    embedded = np.stack([signal[i * tau : i * tau + max_i] for i in range(emb_dim)], axis=1)
    # Pairwise distance matrix (subsample if too large)
    max_pts  = 1000
    if len(embedded) > max_pts:
        idx      = np.linspace(0, len(embedded) - 1, max_pts, dtype=int)
        embedded = embedded[idx]
    diff      = embedded[:, None, :] - embedded[None, :, :]   # (N, N, emb_dim)
    dist      = np.sqrt((diff ** 2).sum(axis=-1)).astype(np.float32)
    threshold = np.percentile(dist, 10)
    rp        = (dist <= threshold).astype(np.float32)
    return resize(rp, (img_size, img_size), anti_aliasing=False).astype(np.float32)

def signal_to_spectrogram(signal, img_size=IMG_SIZE):
    ch1 = compute_stft_channel(signal, img_size)
    ch2 = compute_cwt_channel(signal,  img_size)
    ch3 = compute_recurrence_channel(signal, img_size)
    return np.stack([ch1, ch2, ch3], axis=0)   # (3, 224, 224)

def build_spectrograms(windows, labels, rids):
    n       = len(windows)
    spectros = np.zeros((n, 3, IMG_SIZE, IMG_SIZE), dtype=np.float32)
    for i, sig in enumerate(windows):
        spectros[i] = signal_to_spectrogram(sig)
        if (i + 1) % 500 == 0:
            print(f"  spectrogram {i+1}/{n}")
    return spectros, labels, rids



## Main function

In [ ]:

def main():
    print("=== Phase 1: Data Preprocessing ===\n")

    # ── Load ──
    print("[1/7] Loading data...")
    meta, signals = load_data(DATA_DIR)
    print(f"  Records: {len(meta)}  |  Label distribution:\n{meta['label'].value_counts()}\n")

    # ── Split at record level ──
    print("[2/7] Record-level train/test split (80/20, stratified)...")
    train_mask, test_mask = split_records(meta)
    print(f"  Train records: {train_mask.sum()}  |  Test records: {test_mask.sum()}\n")

    # ── Build windows ──
    print("[3/7] Extracting windows...")
    train_wins, train_labels, train_rids = build_windows(
        meta, signals, train_mask, WINDOW_SIZE, TRAIN_STEP
    )
    test_wins, test_labels, test_rids = build_windows(
        meta, signals, test_mask, WINDOW_SIZE, TEST_STEP
    )
    print(f"  Train windows: {len(train_wins)}  |  Test windows: {len(test_wins)}")
    for lbl, name in [(0,"N"),(1,"A"),(2,"O"),(3,"~")]:
        print(f"    Train {name}: {(train_labels==lbl).sum()}  |  Test {name}: {(test_labels==lbl).sum()}")
    print()

    # ── Augmentation (train only) ──
    print("[4/7] Augmenting train windows (non-Normal only, CutMix + noise)...")
    train_wins, train_labels, train_rids = augment_windows(
        train_wins, train_labels, train_rids, AUG_TARGETS
    )
    print(f"  After augmentation — Train windows: {len(train_wins)}")
    for lbl, name in [(0,"N"),(1,"A"),(2,"O"),(3,"~")]:
        print(f"    {name}: {(train_labels==lbl).sum()}")
    print()

    # ── Save raw windows (Stage 1 input) ──
    print("[5/7] Saving raw windowed signals...")
    # Shape: (N, 4500) — the TCN will add channel dim itself
    np.savez_compressed(
        f"{OUT_DIR}/train_raw.npz",
        windows=train_wins,
        labels=train_labels,
        record_ids=train_rids
    )
    np.savez_compressed(
        f"{OUT_DIR}/test_raw.npz",
        windows=test_wins,
        labels=test_labels,
        record_ids=test_rids
    )
    print("  Saved train_raw.npz and test_raw.npz\n")

    # ── Build spectrograms (Stage 2 input) ──
    # Stage 2 only uses non-Normal windows
    train_nonnormal_mask = train_labels != 0
    test_nonnormal_mask  = test_labels  != 0

    s2_train_wins   = train_wins[train_nonnormal_mask]
    s2_train_labels = train_labels[train_nonnormal_mask]
    s2_train_rids   = train_rids[train_nonnormal_mask]

    s2_test_wins    = test_wins[test_nonnormal_mask]
    s2_test_labels  = test_labels[test_nonnormal_mask]
    s2_test_rids    = test_rids[test_nonnormal_mask]

    # Remap labels for Stage 2: A→0, O→1, ~→2
    s2_label_remap = {1: 0, 2: 1, 3: 2}
    s2_train_labels = np.array([s2_label_remap[l] for l in s2_train_labels], dtype=np.int64)
    s2_test_labels  = np.array([s2_label_remap[l] for l in s2_test_labels],  dtype=np.int64)

    print("[6/7] Generating 3-channel spectrograms (this will take a while)...")
    print("  Processing TRAIN spectrograms...")
    train_spectros, _, _ = build_spectrograms(s2_train_wins, s2_train_labels, s2_train_rids)
    print("  Processing TEST spectrograms...")
    test_spectros,  _, _ = build_spectrograms(s2_test_wins,  s2_test_labels,  s2_test_rids)

    print("[7/7] Saving spectrograms...")
    np.savez_compressed(
        f"{OUT_DIR}/train_spectro.npz",
        spectrograms=train_spectros,
        labels=s2_train_labels,
        record_ids=s2_train_rids
    )
    np.savez_compressed(
        f"{OUT_DIR}/test_spectro.npz",
        spectrograms=test_spectros,
        labels=s2_test_labels,
        record_ids=s2_test_rids
    )
    print("  Saved train_spectro.npz and test_spectro.npz\n")

    # Save a placeholder scaler (will be used for NeuroKit features in Phase 2)
    scaler = StandardScaler()
    with open(f"{OUT_DIR}/scaler.pkl", "wb") as f:
        pickle.dump(scaler, f)
    print("  Saved scaler.pkl (unfitted placeholder for NeuroKit features)\n")

    print("=== Phase 1 complete ===")
    print(f"Output files in '{OUT_DIR}':")
    print("  train_raw.npz     — shape (N, 4500), Stage 1 input")
    print("  test_raw.npz      — shape (N, 4500), Stage 1 input")
    print("  train_spectro.npz — shape (N, 3, 224, 224), Stage 2 input (non-Normal only, labels remapped A→0 O→1 ~→2)")
    print("  test_spectro.npz  — shape (N, 3, 224, 224), Stage 2 input (non-Normal only, labels remapped A→0 O→1 ~→2)")
    print("  scaler.pkl        — StandardScaler placeholder")


In [13]:

if __name__ == "__main__":
    main()

=== Phase 1: Data Preprocessing ===

[1/7] Loading data...
  Records: 8528  |  Label distribution:
label
N    5050
O    2456
A     738
~     284
Name: count, dtype: int64

[2/7] Record-level train/test split (80/20, stratified)...
  Train records: 6822  |  Test records: 1706

[3/7] Extracting windows...
  Train windows: 22083  |  Test windows: 3496
    Train N: 12895  |  Test N: 2042
    Train A: 1856  |  Test A: 307
    Train O: 6900  |  Test O: 1067
    Train ~: 432  |  Test ~: 80

[4/7] Augmenting train windows (non-Normal only, CutMix + noise)...
  After augmentation — Train windows: 31271
    N: 12895
    A: 3712
    O: 13800
    ~: 864

[5/7] Saving raw windowed signals...
  Saved train_raw.npz and test_raw.npz

[6/7] Generating 3-channel spectrograms (this will take a while)...
  Processing TRAIN spectrograms...
  spectrogram 500/18376
  spectrogram 1000/18376
  spectrogram 1500/18376
  spectrogram 2000/18376
  spectrogram 2500/18376
  spectrogram 3000/18376
  spectrogram 3500/1

## Converting npz to np

In [5]:
import numpy as np

data = np.load("New_Processed_data/train_spectro.npz")
np.save("train_spectro_X.npy", data["spectrograms"])
np.save("train_spectro_y.npy", data["labels"])

data = np.load("New_Processed_data/test_spectro.npz")
np.save("test_spectro_X.npy", data["spectrograms"])
np.save("test_spectro_y.npy", data["labels"])